# 01 — Optuna PostgreSQL Orchestrator

Jalankan notebook ini di **kernel sendiri** pada laptop lokal. Notebook ini tidak melakukan training.

Tugasnya:
- membaca `stage_information` dari PostgreSQL lokal `cqcnn_orchestration`;
- membuka Optuna storage di PostgreSQL lokal/shared;
- melakukan `study.ask()` dan `trial.suggest_*`;
- membuat task di PostgreSQL lokal secara rolling;
- menjaga jumlah task aktif mendekati `MAX_PARALLEL_TASKS`.

Jalankan `worker_runner` di kernel lain.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PY_DIR = PROJECT_ROOT / "patched_program_files"
if str(PY_DIR) not in sys.path:
    sys.path.insert(0, str(PY_DIR))

from env_loader import load_dotenv_if_exists, require_env
from postgres_orchestration_db import PostgresOrchestrationDb, PostgresOrchestrationDbConfig
from optuna_stage_manager import run_dispatcher_from_postgres

load_dotenv_if_exists(PROJECT_ROOT / ".env")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# === EDIT PARAMETER INI ===
STAGE_NO = 1
MODEL_TYPE = "classical_fully_spatial"  # "classical_fully_spatial" atau "hybrid_qcqcnn"

# Set sesuai jumlah worker aktif: laptop worker + PC worker lain.
MAX_PARALLEL_TASKS = 10
POLL_SECONDS = 30
SEED = 42
SOURCE_BEST_STAGE_NO = 4  # dipakai saat STAGE_NO = 5

# Dari .env
ORCHESTRATION_DB_DSN = require_env("ORCHESTRATION_DB_DSN")
OPTUNA_STORAGE_URL = require_env("OPTUNA_STORAGE_URL")

print("STAGE_NO:", STAGE_NO)
print("MODEL_TYPE:", MODEL_TYPE)
print("MAX_PARALLEL_TASKS:", MAX_PARALLEL_TASKS)
print("SOURCE_BEST_STAGE_NO:", SOURCE_BEST_STAGE_NO)
print("OPTUNA_STORAGE_URL:", OPTUNA_STORAGE_URL)

In [ ]:
db = PostgresOrchestrationDb(PostgresOrchestrationDbConfig(dsn=ORCHESTRATION_DB_DSN))
print("orchestration connection:", db.test_connection())
print("stage_information rows:", len(db.query_stage_information()))
selected_stage_rows = db.query_stage_information(stage_no=STAGE_NO, model_type=MODEL_TYPE)
selected_stage_rows

In [ ]:
# Run cell ini dan biarkan berjalan untuk stage HPO.
# Untuk Stage 5 non-HPO, cell ini hanya membuat satu task final repeated 5-fold.
dispatch_result = run_dispatcher_from_postgres(
    db=db,
    stage_no=STAGE_NO,
    model_type=MODEL_TYPE,
    optuna_storage_url=OPTUNA_STORAGE_URL,
    max_parallel_tasks=MAX_PARALLEL_TASKS,
    poll_seconds=POLL_SECONDS,
    seed=SEED,
    stop_when_complete=True,
    source_best_stage_no=SOURCE_BEST_STAGE_NO,
)
dispatch_result